In [1]:
import pandas as pd

train = pd.read_parquet('../data/processed/features_train.parquet')
test = pd.read_parquet('../data/processed/features_test.parquet')


In [2]:
NON_FEATURES = ['target', 'issue_d', 'earliest_cr_line', 'zip_code', 'addr_state']

X_train = train.drop(columns=NON_FEATURES)
y_train = train['target']

X_test  = test.drop(columns=NON_FEATURES)
y_test  = test['target']

In [3]:
print(X_train.shape, X_test.shape)
print("train bad rate:", y_train.mean().round(4))
print("test bad rate: ", y_test.mean().round(4))

(20902, 71) (5378, 71)
train bad rate: 0.2005
test bad rate:  0.2365


In [4]:
import numpy as np

for name, X in [("train", X_train), ("test", X_test)]:
    print(f"--- {name} ---")
    print("shape:        ", X.shape)
    print("missing:      ", X.isna().sum().sum())
    print("inf:          ", np.isinf(X).sum().sum())
    print("constant cols:", X.columns[X.nunique() == 1].tolist())
    print()

--- train ---
shape:         (20902, 71)
missing:       0
inf:           0
constant cols: []

--- test ---
shape:         (5378, 71)
missing:       0
inf:           0
constant cols: []



In [5]:
from sklearn.preprocessing import StandardScaler

# the scaler works by position, so the two halves must line up exactly
assert list(X_train.columns) == list(X_test.columns), "column order differs"

scaler = StandardScaler()
scaler.fit(X_train)          # means and stds learned from train only

X_train_s = pd.DataFrame(
    scaler.transform(X_train),
    columns=X_train.columns,
    index=X_train.index,
)
X_test_s = pd.DataFrame(
    scaler.transform(X_test),   # train's numbers, applied to test
    columns=X_test.columns,
    index=X_test.index,
)

print("train  mean:", round(X_train_s.mean().mean(), 6),
      " std:", round(X_train_s.std().mean(), 4))
print("test   mean:", round(X_test_s.mean().mean(), 4),
      " std:", round(X_test_s.std().mean(), 4))

train  mean: -0.0  std: 1.0
test   mean: -0.0066  std: 1.0813


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = LogisticRegression(
    max_iter=2000,      # high enough that the solver settles; watch for a warning
    random_state=42,    # same run every time
)
model.fit(X_train_s, y_train)   # scaled features, not the originals

# column 1 is P(default); column 0 is P(no default)
p_train = model.predict_proba(X_train_s)[:, 1]
p_test  = model.predict_proba(X_test_s)[:, 1]

auc_train = roc_auc_score(y_train, p_train)
auc_test  = roc_auc_score(y_test,  p_test)

print("train AUC:", round(auc_train, 4))
print("test  AUC:", round(auc_test, 4))
print("gap:      ", round(auc_train - auc_test, 4))

train AUC: 0.7156
test  AUC: 0.6975
gap:       0.0181


In [7]:
dummy_probs = np.full(len(y_test), 0.2005)
dummy_probs

dummy_auc = roc_auc_score(y_test, dummy_probs)
print('dummy_auc : ', round(dummy_auc, 4))

dummy_auc :  0.5


In [8]:
# ---- 1. the no-skill baseline: same score for every loan ----------------
p_dumb = np.full(len(y_test), y_train.mean())
auc_dumb = roc_auc_score(y_test, p_dumb)

print("dumb baseline AUC:", round(auc_dumb, 4))
print("our model AUC:    ", round(auc_test, 4))
print()

# ---- 2. the weights ------------------------------------------------------
weights = pd.DataFrame({
    "feature": X_train_s.columns,      # same order as coef_
    "weight":  model.coef_[0],         # [0] unwraps the single row
})
weights["size"] = weights["weight"].abs()
weights = weights.sort_values("size", ascending=False)

print("--- top 15 (model leans on these most) ---")
print(weights.head(15).to_string(index=False))
print()
print("--- bottom 5 (model barely used these) ---")
print(weights.tail(5).to_string(index=False))

dumb baseline AUC: 0.5
our model AUC:     0.6975

--- top 15 (model leans on these most) ---
                   feature    weight     size
                      term  0.411181 0.411181
         total_bal_ex_mort  0.324083 0.324083
total_il_high_credit_limit -0.304806 0.304806
          total_rev_hi_lim -0.273828 0.273828
                       dti  0.227880 0.227880
       num_rev_tl_bal_gt_0  0.227150 0.227150
      mo_sin_old_rev_tl_op -0.216047 0.216047
      acc_open_past_24mths  0.210015 0.210015
     credit_history_months  0.176430 0.176430
                      fico -0.158656 0.158656
            loan_to_income  0.143742 0.143742
     mths_since_recent_inq -0.139418 0.139418
             log_revol_bal -0.127879 0.127879
            bc_open_to_buy  0.126251 0.126251
           tot_hi_cred_lim  0.123070 0.123070

--- bottom 5 (model barely used these) ---
                   feature    weight     size
purpose_debt_consolidation  0.001831 0.001831
    purpose_major_purchase -0.00181

In [9]:
top15 = weights['feature'].head(15)
X_train_s[top15].corr()

,term,total_bal_ex_mort,total_il_high_credit_limit,total_rev_hi_lim,dti,num_rev_tl_bal_gt_0,mo_sin_old_rev_tl_op,acc_open_past_24mths,credit_history_months,fico,loan_to_income,mths_since_recent_inq,log_revol_bal,bc_open_to_buy,tot_hi_cred_lim
term,1.000000,0.103795,0.087022,0.059659,0.077083,0.053994,0.022357,0.061061,0.021068,-0.005777,0.327563,-0.038182,0.113318,-0.000870,0.100122
total_bal_ex_mort,0.103795,1.000000,0.869369,0.420265,0.292879,0.143927,0.111497,0.151589,0.123717,0.041379,-0.085708,-0.076913,0.333805,0.096895,0.492886
total_il_high_credit_limit,0.087022,0.869369,1.000000,0.089955,0.326593,0.018949,0.021286,0.186190,0.041261,0.056987,-0.126311,-0.103610,0.093925,0.031182,0.381572
total_rev_hi_lim,0.059659,0.420265,0.089955,1.000000,0.077653,0.327904,0.282841,0.080991,0.259195,0.325722,0.029805,-0.007175,0.538901,0.632805,0.489516
dti,0.077083,0.292879,0.326593,0.077653,1.000000,0.255314,0.039134,0.166881,0.037900,-0.055897,0.252773,-0.049456,0.214817,-0.075937,-0.003107
num_rev_tl_bal_gt_0,0.053994,0.143927,0.018949,0.327904,0.255314,1.000000,0.165227,0.321118,0.156092,-0.162286,0.054625,-0.092414,0.452848,0.107470,0.110645
mo_sin_old_rev_tl_op,0.022357,0.111497,0.021286,0.282841,0.039134,0.165227,1.000000,-0.052163,0.920116,0.125303,0.003264,0.023023,0.221710,0.186152,0.205690
acc_open_past_24mths,0.061061,0.151589,0.186190,0.080991,0.166881,0.321118,-0.052163,1.000000,-0.032333,-0.101935,-0.079116,-0.329334,-0.005462,0.087173,0.075476
credit_history_months,0.021068,0.123717,0.041261,0.259195,0.037900,0.156092,0.920116,-0.032333,1.000000,0.109632,-0.015217,0.012491,0.200649,0.167224,0.204819
fico,-0.005777,0.041379,0.056987,0.325722,-0.055897,-0.162286,0.125303,-0.101935,0.109632,1.000000,0.030598,0.118324,-0.031156,0.514755,0.210546


In [10]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ---- 1. correlation among the top 15 features ---------------------------
top15 = weights.head(15)["feature"].tolist()

corr = X_train_s[top15].corr()

# keep each pair once: upper triangle only, diagonal skipped
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
pairs = corr.where(mask).unstack().dropna()

strong = pairs[pairs.abs() > 0.7].sort_values(key=abs, ascending=False)

print("--- pairs correlated above 0.7 (top 15 features) ---")
if len(strong) == 0:
    print("none")
else:
    for (a, b), v in strong.items():
        print(f"{v:+.3f}   {a}  <->  {b}")
print()

# ---- 2. VIF across all 71 features --------------------------------------
arr = X_train_s.values   # the function wants a plain array

vif = pd.DataFrame({
    "feature": X_train_s.columns,
    "vif": [variance_inflation_factor(arr, i) for i in range(arr.shape[1])],
}).sort_values("vif", ascending=False)

print("--- top 15 by VIF ---")
print(vif.head(15).to_string(index=False))

--- pairs correlated above 0.7 (top 15 features) ---
+0.920   credit_history_months  <->  mo_sin_old_rev_tl_op
+0.869   total_il_high_credit_limit  <->  total_bal_ex_mort

--- top 15 by VIF ---
                   feature        vif
                  open_acc 470.738890
                  num_sats 465.367753
                 total_acc 356.547415
             num_rev_accts 171.496342
                 num_il_tl 132.025884
           tot_hi_cred_lim  91.568107
               tot_cur_bal  88.165555
       num_rev_tl_bal_gt_0  42.964596
           num_actv_rev_tl  40.614631
          total_rev_hi_lim  34.042106
purpose_debt_consolidation  31.052967
       purpose_credit_card  23.406439
                 revol_bal  21.580779
         total_bal_ex_mort  19.441993
total_il_high_credit_limit  16.927757


In [11]:
assert train['issue_d'].is_monotonic_increasing

In [16]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

assert train["issue_d"].is_monotonic_increasing, "train rows are not in time order"

tscv = TimeSeriesSplit(n_splits=5)

model_l2 = LogisticRegression(max_iter=2000, random_state=42)
model_l1 = LogisticRegression(l1_ratio=1.0, solver="liblinear",
                              max_iter=2000, random_state=42)

scores_l2 = cross_val_score(model_l2, X_train_s, y_train, cv=tscv, scoring="roc_auc")
scores_l1 = cross_val_score(model_l1, X_train_s, y_train, cv=tscv, scoring="roc_auc")

print(f"L2  mean AUC: {scores_l2.mean():.4f}   std: {scores_l2.std():.4f}")
print(f"L1  mean AUC: {scores_l1.mean():.4f}   std: {scores_l1.std():.4f}")
print(f"difference:   {abs(scores_l2.mean() - scores_l1.mean()):.4f}")
print()
print("per fold L2:", np.round(scores_l2, 4))
print("per fold L1:", np.round(scores_l1, 4))
print()

model_l1.fit(X_train_s, y_train)
n_zero = (model_l1.coef_[0] == 0).sum()
n_kept = (model_l1.coef_[0] != 0).sum()
print(f"L1 zeroed {n_zero} features, kept {n_kept} of {X_train_s.shape[1]}")

L2  mean AUC: 0.7046   std: 0.0168
L1  mean AUC: 0.7060   std: 0.0175
difference:   0.0014

per fold L2: [0.6797 0.7091 0.7147 0.7273 0.6922]
per fold L1: [0.6806 0.7114 0.7159 0.7299 0.6923]

L1 zeroed 3 features, kept 68 of 71


In [17]:
C_VALUES = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 10.0]

rows = []
for C in C_VALUES:
    m = LogisticRegression(l1_ratio=1.0, solver="liblinear", C=C,
                           max_iter=2000, random_state=42)

    scores = cross_val_score(m, X_train_s, y_train, cv=tscv, scoring="roc_auc")

    m.fit(X_train_s, y_train)              # full train, just to count survivors
    kept = (m.coef_[0] != 0).sum()

    rows.append({"C": C,
                 "mean_auc": scores.mean(),
                 "std": scores.std(),
                 "kept": kept})

    print(f"C={C:<7} auc={scores.mean():.4f}  kept={kept}")   # progress

print()
sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

# the decision rule: smallest model still within one std of the best score
best = sweep.loc[sweep["mean_auc"].idxmax()]
floor = best["mean_auc"] - best["std"]
ok = sweep[sweep["mean_auc"] >= floor]

print()
print(f"best:  C={best['C']}  auc={best['mean_auc']:.4f}  kept={int(best['kept'])}")
print(f"within one std (>= {floor:.4f}):")
print(ok.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

C=0.001   auc=0.5396  kept=1
C=0.005   auc=0.6577  kept=20
C=0.01    auc=0.7000  kept=33
C=0.05    auc=0.7087  kept=47
C=0.1     auc=0.7089  kept=56
C=0.5     auc=0.7074  kept=65
C=1.0     auc=0.7060  kept=68
C=10.0    auc=0.7026  kept=71

      C  mean_auc    std  kept
 0.0010    0.5396 0.0526     1
 0.0050    0.6577 0.0810    20
 0.0100    0.7000 0.0201    33
 0.0500    0.7087 0.0167    47
 0.1000    0.7089 0.0166    56
 0.5000    0.7074 0.0168    65
 1.0000    0.7060 0.0175    68
10.0000    0.7026 0.0189    71

best:  C=0.1  auc=0.7089  kept=56
within one std (>= 0.6924):
      C  mean_auc    std  kept
 0.0100    0.7000 0.0201    33
 0.0500    0.7087 0.0167    47
 0.1000    0.7089 0.0166    56
 0.5000    0.7074 0.0168    65
 1.0000    0.7060 0.0175    68
10.0000    0.7026 0.0189    71


In [18]:
model_final = LogisticRegression(l1_ratio=1.0, solver="liblinear", C=0.05,
                                 max_iter=2000, random_state=42)
model_final.fit(X_train_s, y_train)

coefs = pd.Series(model_final.coef_[0], index=X_train_s.columns)

print(f"kept {(coefs != 0).sum()}, dropped {(coefs == 0).sum()}")
print()

# did the overlapping pairs get resolved?
pairs = [("open_acc", "num_sats"),
         ("credit_history_months", "mo_sin_old_rev_tl_op"),
         ("total_il_high_credit_limit", "total_bal_ex_mort"),
         ("tot_hi_cred_lim", "total_rev_hi_lim")]

for a, b in pairs:
    print(f"{a:28} {coefs[a]:+.4f}    {b:28} {coefs[b]:+.4f}")
print()

print("--- dropped ---")
print(sorted(coefs[coefs == 0].index.tolist()))

kept 47, dropped 24

open_acc                     +0.0000    num_sats                     +0.0000
credit_history_months        +0.1005    mo_sin_old_rev_tl_op         -0.1368
total_il_high_credit_limit   -0.0806    total_bal_ex_mort            +0.0912
tot_hi_cred_lim              -0.0267    total_rev_hi_lim             -0.0247

--- dropped ---
['acc_now_delinq', 'annual_inc', 'application_type_Joint App', 'bc_open_to_buy', 'chargeoff_within_12_mths', 'emp_length', 'log_annual_inc', 'mo_sin_old_il_acct', 'mo_sin_rcnt_rev_tl_op', 'num_actv_rev_tl', 'num_bc_tl', 'num_il_tl', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_90g_dpd_24m', 'open_acc', 'open_to_total_acc', 'pct_tl_nvr_dlq', 'pub_rec', 'purpose_debt_consolidation', 'purpose_major_purchase', 'revol_bal', 'tax_liens', 'tot_cur_bal']
